(large:GPS)=
# Projekt: GPS og geometri – lineære og ikke-lineære ligninger

*Baseret på original køreplan til 01005 Matematik 1, DTU. Opdateret til Python og Mat1a+1b af Adam Greve Bregenhof*

## Indledning

Hvordan ved din telefon præcis, hvor du er? Bag Google Maps og selvkørende biler ligger et netværk af ca. 24 satellitter og et fascinerende matematisk problem: GPS-positionering handler nemlig helt fundamentalt om at løse ligningssystemer, hvor ukendte koordinater og tidsfejl skal findes ud fra signaler der bevæger sig med lysets hastighed.

I dette projekt skal du dykke ned i den geometri og lineære algebra, der gør global navigation mulig. Vi starter med det ideelle tilfælde: Hvor skærer tre cirkler (eller kugler) hinanden? Men virkeligheden er dog mere kompleks. Satellitternes ure er ekstremt præcise atomure, mens dit ur i telefonen er billigt og upræcist. Denne lille tidsforskel skaber en fjerde ubekendt, som kræver en fjerde satellit og et mere avanceret system af ligninger.

Undervejs vil du arbejde med:

* **Afstandsligninger:** Opstilling af ikke-lineære ligningssystemer baseret på Pythagoras i 2D og 3D.
* **Linearisering:** Hvordan man transformerer komplekse cirkelligninger til simple lineære ligningssystemer ved hjælp af subtraktion.
* **Iterative metoder:** Hvordan computere finder løsninger, når vi ikke kan isolere $x$ og $y$ direkte.
* **Python-implementering:** Du skal bygge din egen "GPS-modtager" i kode, der kan beregne en position ud fra rå satellitdata og håndtere urfejl.

Projektet kombinerer klassisk geometri med moderne numeriske metoder.

### Forberedelse

Inden projektet anbefales det at have set på følgende stof:

* Vektorer og koordinater i 2D og 3D (Mat1a, kapitel 1–2)
* Lineære ligningssystemer og matrix-ligninger $A\mathbf{x} = \mathbf{b}$ (Mat1a, kapitel 3)
* Symmetrisk matrix, positiv definit matrix (Mat1b, kapitel 4)
* Taylor-udvikling af funktioner af flere variable (Mat1b, kapitel 6)

Endelig kan man med fordel også læse mere om mindste kvadraters metode og normalligningerne $A^T A \mathbf{x} = A^T \mathbf{b}$
her: https://data.math.au.dk/interactive/lt/mindstekvadrater.html. Metoden er måske allerede kendt fra lineær regression.

### Projektmål

Målet med projektet er at forstå den matematiske model bag GPS-positionering og implementere en komplet løsningsalgoritme i Python. Du vil:

1. Opstille ikke-lineære observationsligninger fra geometriske betragtninger.
2. Linearisere disse ligninger vha. Taylor-udvikling.
3. Implementere en iterativ løsningsmetode (Gauss-Newtons metode).
4. Udvide til overbestemte systemer med mindste kvadraters metode.

```{note}
I skal skrive en sammenhængende rapport uden nødvendigvis at besvare alle opgavenumrene nedenfor. Det forventes ikke, at alle opgaver besvares. I skal lade den endelige rapport styre af jeres interesser og ambitioner.

Opgaver markeret med (*) er valgfrie i den endelige rapport. Opgaver med (**) er særligt udfordrende og er tiltænkt interesserede.
```

## Opsætning

Følgende pakker skal installeres og importeres

In [41]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from mpl_toolkits.mplot3d import Axes3D
from scipy.linalg import solve
from scipy.optimize import fsolve
from itertools import combinations

Nedenfor defineres de faste konstanter og de satellitdata, der bruges i resten af projektet.

In [42]:
# Lysets hastighed i m/s
c = 299_792_458

# Satellit-data: [X_s, Y_s, Z_s, pseudoafstand P_s]  (alle i meter)
satellitter = np.array([
    [ 4396623.907, -15219512.421,  21395963.449,  22745185],  # Sat 4
    [12508988.544,  10190642.686,  21073725.487,  20490898],  # Sat 14
    [15690450.142,  -6024729.548,  20505506.585,  20673780],  # Sat 16
    [25180700.379,  -7388281.420,   4051959.985,  23193809],  # Sat 18
    [-8441248.473, -20422306.240,  15063403.391,  26237952],  # Sat 24
    [-2062043.991,  17167206.349,  20373015.170,  22979470],  # Sat 25
])

# Navne på satellitterne
sat_navne = [4, 14, 16, 18, 24, 25]

# Foreløbig startposition (X0, Y0, Z0)
x0 = np.array([3504300.0, 780800.0, 5252100.0])

# Kendt sand position
x_sand = np.array([3504320.6, 780753.5, 5252128.8])

print("Satellitdata indlæst.")
print(f"Startposition:  X={x0[0]:.1f}, Y={x0[1]:.1f}, Z={x0[2]:.1f} m")
print(f"Sand position:  X={x_sand[0]:.1f}, Y={x_sand[1]:.1f}, Z={x_sand[2]:.1f} m")

Satellitdata indlæst.
Startposition:  X=3504300.0, Y=780800.0, Z=5252100.0 m
Sand position:  X=3504320.6, Y=780753.5, Z=5252128.8 m


---

## Modul 1 – Signalet: Fra transmissionstid til afstand

### Baggrund

GPS-systemet består af ca. 24 satellitter i kredsløb om Jorden. Hver satellit udsender et radiosignal med information om sit eget tidsstempel og sin position. En GPS-modtager måler, hvor lang tid signalet er om at ankomme, og omregner denne tid til en afstand:

$$
P = c \cdot \Delta t
$$

hvor $c \approx 2{,}998 \times 10^8\ \text{m/s}$ er lysets hastighed og $\Delta t$ er transmissionstiden.

Denne afstand kaldes en **pseudoafstand** (ikke en rigtig geometrisk afstand), fordi modtagerens ur ikke er perfekt synkroniseret med satellittens atomur. Vi vender tilbage til denne komplikation i Modul 4.

Alle beregninger foregår i koordinatsystemet **WGS84** (World Geodetic System 1984): et tredimensionalt kartesisk system med origo i Jordens massemidtpunkt, $x$-aksen mod skæringen af Ækvatoren og Greenwichmeridianen, og $z$-aksen mod Nordpolen.

### Opgave 1 – Tidsfejl og afstandsfejl

Selv en lille fejl i modtagerens ur giver en stor fejl i positionsbestemmelsen.

**a)** En urfejl på $\Delta t = 1\ \text{ms}$ giver en afstandsfejl på ca. 300 km. Bekræft dette ved at beregne $c \cdot \Delta t$ for $\Delta t = 10^{-3}\ \text{s}$.

**b)** Beregn den maksimale tilladte urfejl $\Delta t_{\max}$, hvis vi ønsker en positionsfejl på under 10 meter.

**c)** Et almindeligt kvartsur har en nøjagtighed på ca. $\pm 15\ \text{ms/dag}$. Beregn, hvor stor en positionsfejl dette svarer til per dag. Forklar ud fra din beregning, hvorfor urfejlen altid behandles som en ubekendt i GPS-systemet.

**d)** WGS84-koordinatsystemet er et **højredrejet** kartesisk koordinatsystem. Vi ved, at:
* $x$-aksen peger mod skæringen af Ækvatoren og Greenwichmeridianen (0°N, 0°Ø),
* $z$-aksen peger mod den geografiske Nordpol.

Brug definitionen på et højredrejet koordinatsystem til at bestemme retningen for $y$-aksen. Udregn hvilken bredde- og længdegrad $y$-aksen stikker ud af jordoverfladen ved, og find det tilsvarende sted på et verdenskort. Beskriv kort, hvad der geografisk befinder sig der.

In [3]:
print(f"a) {(c * 10e-3)/1000} km")
d_max = 10
delta_t_max_10m = d_max/c
print(f"b) {delta_t_max_10m} s")
print(f"c) {(c*15e-3)/1000} km")


a) 2997.92458 km
b) 3.3356409519815205e-08 s
c) 4496.88687 km


***Opgave 1a

In [4]:
delta_t = 0.001
c * delta_t 

299792.458

Opgave 1b

In [5]:
10 / c

3.3356409519815205e-08

Opgave 1c

In [6]:
delta_t_lower = -0.0015
delta_t_upper = 0.0015
pos_unc_lower = delta_t_lower * c
pos_unc_upper = delta_t_upper * c
print(pos_unc_lower,pos_unc_upper)

-449688.68700000003 449688.68700000003


Det er en ubekendt da positionsfejlen kan antage hvilken som helst værdi inden for dette interval.

Opgave 1d

Vi tænker at koordinatsystemet starter inde i midten af jorden og ved brug af højrehåndsreglen vil det placere y-aksen ortogonalt til x- og z-akserne, i retning mod uret i forhold til nordpolen, det betyder at y-aksen stikker ud ved breddegrad 0 og længdegrad 90, hvilket svarer til det indiske ocean.

---

## Modul 2 – 2D-geometri og lineariseringstricket

### Baggrund

Inden vi går til den fulde 3D-model, studerer vi det simplere 2D-tilfælde – uden urfejl. Her kender vi vores afstand til tre kendte punkter og ønsker at finde vores position.

Hvis en modtager befinder sig i $(x, y)$, og vi kender afstanden $r_i$ til tre punkter $(x_i, y_i)$, gælder:

$$\begin{cases}
(x - x_1)^2 + (y - y_1)^2 = r_1^2 \\
(x - x_2)^2 + (y - y_2)^2 = r_2^2 \\
(x - x_3)^2 + (y - y_3)^2 = r_3^2
\end{cases}$$

Geometrisk svarer dette til at finde skæringen af tre cirkler (hvis den findes). Ligningerne er ikke-lineære (kvadratiske i $x$ og $y$), men der er et elegant trick til at linearisere dem. Sådanne algebraiske reduktioner vil vi se igen senere i projektet.

### Opgave 2 – Linearisering ved subtraktion

**a)** Udvid ligning 1 og ligning 2. Træk ligning 1 fra ligning 2. Vis, at de kvadratiske led $x^2$ og $y^2$ forsvinder, og at du får en lineær ligning i $x$ og $y$ på formen $\alpha x + \beta y = \gamma$.

**b)** Gør det samme for ligning 1 trukket fra ligning 3. Du har nu to lineære ligninger med to ubekendte. Opstil det lineære system $A\mathbf{x} = \mathbf{b}$ med:

$$ 
A = \begin{pmatrix} 2(x_2-x_1) & 2(y_2-y_1) \\ 2(x_3-x_1) & 2(y_3-y_1) \end{pmatrix}, \quad \mathbf{b} = \begin{pmatrix} r_1^2-r_2^2+(x_2^2-x_1^2)+(y_2^2-y_1^2) \\ r_1^2-r_3^2+(x_3^2-x_1^2)+(y_3^2-y_1^2) \end{pmatrix} 
$$

**c)** Implementér en Python-funktion, der løser dette system for et givet sæt af tre punkter og afstande. Test den med data fra nedenstående eksempel.

Opgave 2a

$$
2x(x_2 - x_1) + 2y(y_2 - y_1) = r_1^2 - r_2^2 + y_2^2 - y_1^2 + x_2^2 - x_1^2
$$

Opgave 2b

$$
2x(x_3 - x_1) + 2y(y_3 - y_1) = r_1^2 - r_3^2 + y_3^2 - y_1^2 + x_3^2 - x_1^2
$$

In [7]:
# Testdata: tre "sendere" i 2D og en modtager i (3, 4)
sendere_2d = np.array([[0.0, 0.0],
                        [6.0, 0.0],
                        [3.0, 7.0]])
modtager_sand_2d = np.array([3.0, 4.0])

# Beregn de "sande" afstande
afstande_2d = np.array([np.linalg.norm(modtager_sand_2d - s) for s in sendere_2d])
print("Afstande fra modtager til senderne:")
for i, (s, r) in enumerate(zip(sendere_2d, afstande_2d)):
    print(f"  r_{i+1} = {r:.4f} m  (til punkt {s})")

Afstande fra modtager til senderne:
  r_1 = 5.0000 m  (til punkt [0. 0.])
  r_2 = 5.0000 m  (til punkt [6. 0.])
  r_3 = 3.0000 m  (til punkt [3. 7.])


In [8]:
def afstand_satellit(sendere, afstande):
    x1, y1 = sendere[0]
    x2, y2 = sendere[1]
    x3, y3 = sendere[2]

    r1, r2, r3 = afstande

    A = np.array([[2*(x2 - x1), 2*(y2 - y1)],
                  [2*(x3 - x1), 2*(y3 - y1)]])
    b = np.array([[r1**2 - r2**2 + (x2**2 - x1**2) + (y2**2 - y1**2)],
                  [r1**2 - r3**2 + (x3**2 - x1**2) + (y3**2 - y1**2)]])
    
    pos = np.linalg.solve(A, b)
    return pos

afstand_satellit(sendere_2d, np.array([5, 5, 3]))

array([[3.],
       [4.]])

### Opgave 3 – Geometrisk fortolkning (*)

**a)** Vis, at de tre linier $L_{12}$, $L_{13}$ og $L_{23}$ (fundet ved parvis subtraktion af cirkelligningerne) alle skærer hinanden i ét fælles punkt $P_C$. Bevis dette algebraisk.

**b)** Hvad repræsenterer linien $L_{ij}$ geometrisk, når cirklerne $i$ og $j$ skærer hinanden i to punkter? Hvad sker der, når de ikke skærer hinanden?

**c) (**)** Vis, at $P_C$ kan bestemmes ved at indføre hjælpevariablen $s = x^2 + y^2$ og løse det resulterende lineære system i $(x, y, s)$.

3a
$$
\begin{aligned}
L_{12} &= 2x(x_2 - x_1) + 2y(y_2 - y_1) = r_2^2 - r_1^2 + x_2^2 - x_1^2 + y_2^2 - y_1^2 \\
L_{13} &= 2x(x_3 - x_1) + 2y(y_3 - y_1) = r_3^2 - r_1^2 + x_3^2 - x_1^2 + y_3^2 - y_1^2 \\
L_{23} &= 2x(x_3 - x_2) + 2y(y_3 - y_2) = r_3^2 - r_2^2 + x_3^2 - x_2^2 + y_3^2 - y_2^2
\end{aligned}
$$

$$
2x(x_2 - x_1) + 2y(y_2 - y_1) - r_2^2 + r_1^2 - x_2^2 + x_1^2 - y_2^2 + y_1^2 = 0
$$

$$
2x(x_3 - x_1) + 2y(y_3 - y_1) - r_3^2 + r_1^2 - x_3^2 + x_1^2 - y_3^2 + y_1^2 = 0
$$

$$
2x(x_3 - x_2) + 2y(y_3 - y_2) - r_3^2 + r_2^2 - x_3^2 + x_2^2 - y_3^2 + y_2^2 = 0
$$

---

3a

$$
C_i (x, y) = (x - x_i)^2 + (y - y_i)^2 - r_i ^2
$$

Et punkt $(x, y)$ ligger på cirklen $i$ hvis $C_i (x, y) = 0$.

$$
L_{ij} \ : \ C_i (x, y) - C_j (x, y) = 0 \implies C_i (x, y) = C_j(x, y)
$$

Hvis antager at linjerne $L_{12}$ og $L_{13}$ skærer hinanden i et punkt $P_C$. $P_C$ ligger på begge de to linjer, derfor må der gælde følgende:

$$
C_1(P_C) = C_2(P_C)
$$

$$
C_1(P_C) = C_3(P_C)
$$

Derfor må det være sandt at

$$
C_2 (P_C) = C_3 (P_C) \implies C_2 (P_C) - C_3 (P_C) = 0
$$

Hvilket er definitionen på linjen $L_{23}$

---

3b

Linjen $L_{ij}$ er potenslinzen for de to cirkler. Det er en linje der altid står vinkelret på den linje der forbinder to cirklers centre.

- Når cirklerene skærer hinanden i to punkter går linjen gennem disse to skæringspunkter.

- Når cirklerne ikke skærer hinanden eksisterer linjen stadig. Set geometrisk er det det sted sted for alle de punkter hvor tangenterne trukket til begge cirkler har samme længde.

---
3c

$$
x^2 - 2x_i x + x_i ^2 + y^2 - 2y_i y + y_i ^2 = r_i ^2
$$

$$
-2x_i x - 2y_i y + s = r_i ^2 - x_i ^2 - y_i ^2
$$

Kan stilles på matrix form

$$
\begin{bmatrix}
-2x_1 & -2y_1 & 1 \\
-2x_2 & -2y_2 & 1 \\
-2x_3 & -2y_3 & 1 
\end{bmatrix}

\begin{bmatrix}
x \\
y \\
s
\end{bmatrix}

=

\begin{bmatrix}
r_1 ^2 - x_1 ^2 - y_1 ^2 \\
r_2 ^2 - x_2 ^2 - y_2 ^2 \\
r_3 ^2 - x_1 ^3 - y_3 ^2 
\end{bmatrix}
$$

Hvis vi udfører rref og ganger med $-1$ får vi følgende:

$$
2(x_2 - x_1)x + 2(y_2 - y_1)y = r_1 ^2 - r_2 ^2 + x_2 ^2 - x_1 ^2 + y_2 ^2 - y_1 ^2
$$

Hvilket er ligningen for $L_{12}$

---

## Modul 3 – Kuglemetoden: Udvidelse til 3D

### Baggrund

I den virkelige verden befinder GPS-satelliterne sig i tre dimensioner. En afstandsmåling til satellit $i$ med position $(X_i, Y_i, Z_i)$ svarer geometrisk til, at modtageren $(X, Y, Z)$ befinder sig på en *kugle* med centrum $(X_i, Y_i, Z_i)$ og radius $r_i$:

$$
(X - X_i)^2 + (Y - Y_i)^2 + (Z - Z_i)^2 = r_i^2.
$$

To kugler skærer i en cirkel, tre kugler skærer typisk i to punkter, og fire kugler giver (ideelt set) et entydigt punkt. I praksis kan man normalt forkaste det ene skæringspunkt, da det enten befinder sig inde i Jorden eller langt ude i rummet.

Lineariseringstricket fra 2D virker stadig: ved at trække ligninger fra hinanden forsvinder de kvadratiske led.

### Opgave 4 – GPS i 3D uden urfejl

**a)** Vis, at 3D-cirkelligningerne kan lineariseres ved parvis subtraktion, analogt til 2D-tilfældet. Opstil det lineære system $A\mathbf{x} = \mathbf{b}$, hvor $\mathbf{x} = (X, Y, Z)^T$.

**b)** Implementér en Python-funktion, der løser 3D-positioneringsproblemet uden urfejl, og test den på de sande satellitpositioner (søjler 1-3 i satellitter-matricen), idet du *erstatter* pseudoafstandene med de sande geometriske afstande til den kendte modtagerposition $x_{sand}$.

In [9]:
# Beregn sande geometriske afstande fra x_sand til satellitterne
geo_afstande = np.array([np.linalg.norm(x_sand - sat[:3]) for sat in satellitter])
print("Sande geometriske afstande til satelliterne (m):")
for i, (navn, r) in enumerate(zip(sat_navne, geo_afstande)):
    print(f"  Satellit {navn}: {r:.3f} m")

Sande geometriske afstande til satelliterne (m):
  Satellit 4: 22747046.224 m
  Satellit 14: 20492754.354 m
  Satellit 16: 20675635.095 m
  Satellit 18: 23195667.204 m
  Satellit 24: 26239807.010 m
  Satellit 25: 22981328.429 m


## Opgave 4a

Parenteserne ophæves:

$$
(X-X_j)^2 + (Y-Y_j)^2 + (Z-Z_j)^2 = r_j^2
$$

$$
X^2 + X_j^2 - 2X_j X + Y^2 + Y_j^2 - 2Y_j Y + Z^2 + Z_j^2 - 2Z_j Z = r_j^2
$$

$$
(X-X_i)^2 + (Y-Y_i)^2 + (Z-Z_i)^2 = r_i^2
$$

$$
X^2 + X_i^2 - 2X_i X + Y^2 + Y_i^2 - 2Y_i Y + Z^2 + Z_i^2 - 2Z_i Z = r_i^2
$$

$$
(X^2 + X_i^2 - 2X_i X + Y^2 + Y_i^2 - 2Y_i Y + Z^2 + Z_i^2 - 2Z_i) Z - X^2 + X_j^2 - 2X_j X + Y^2 + Y_j^2 - 2Y_j Y + Z^2 + Z_j^2 - 2Z_j Z = r_j^2
$$

Opgave 4b

In [10]:
# Denne funktion løser opgaven med 4 satelitter direkte
def afstand_satellit(sat_3d, afstande):
    x1, y1, z1 = sat_3d[0][:3]
    x2, y2, z2 = sat_3d[1][:3]
    x3, y3, z3 = sat_3d[2][:3]
    x4, y4, z4 = sat_3d[3][:3]

    P1 = afstande[0]
    P2 = afstande[1]
    P3 = afstande[2]
    P4 = afstande[3]

    A = np.array([
        [2*(x2 - x1), 2*(y2 - y1), 2*(z2 - z1), 2*(P1 - P2)],
        [2*(x3 - x1), 2*(y3 - y1), 2*(z3 - z1), 2*(P1 - P3)],
        [2*(x4 - x1), 2*(y4 - y1), 2*(z4 - z1), 2*(P1 - P4)]
    ])

    b = np.array([
        P1**2 - P2**2 + (x2**2 - x1**2) + (y2**2 - y1**2) + (z2**2 - z1**2),
        P1**2 - P3**2 + (x3**2 - x1**2) + (y3**2 - y1**2) + (z3**2 - z1**2),
        P1**2 - P4**2 + (x4**2 - x1**2) + (y4**2 - y1**2) + (z4**2 - z1**2)
    ])

    sol = np.linalg.lstsq(A, b, rcond=None)[0]
    return sol[:3], sol[3]

result = afstand_satellit(satellitter[:4, :3], satellitter[:4, 3])
print(f"Position: {result[0]}")
print(f"Clock bias w: {result[1]:.1f} m")

Position: [3194959.03964578  715575.50853473 4805833.44196265]
Clock bias w: 1782261.0 m


In [11]:
# Verify the solution
position_error = np.linalg.norm(result[0] - x_sand)
print(f"True position:  {x_sand}")
print(f"Computed pos:   {result[0]}")
print(f"Position error: {position_error:.2f} m")
print(f"\nClock bias w = {result[1]:.1f} m")
print(f"Individual ur_fejl from satellites: 1855-1861 m (average ~1857 m)")


True position:  [3504320.6  780753.5 5252128.8]
Computed pos:   [3194959.03964578  715575.50853473 4805833.44196265]
Position error: 546929.88 m

Clock bias w = 1782261.0 m
Individual ur_fejl from satellites: 1855-1861 m (average ~1857 m)


In [ ]:
# Denne funktion løser problemet ved at bruge least squares.
def afstand_satellit_lstsq(sat_3d, afstande):

    x1, y1, z1 = sat_3d[0][:3]
    x2, y2, z2 = sat_3d[1][:3]
    x3, y3, z3 = sat_3d[2][:3]
    x4, y4, z4 = sat_3d[3][:3]

    r1 = afstande[0]
    r2 = afstande[1]
    r3 = afstande[2]
    r4 = afstande[3]


    A = np.array([[2*(x2 - x1), 2*(y2 - y1), 2*(z2 - z1)],
                  [2*(x3 - x1), 2*(y3 - y1), 2*(z3 - z1)]
                  [2*(x4 - x1), 2*(y4 - y1), 2*(z4 - z1)]])
    b = np.array([[r1**2 - r2**2 + (x2**2 - x1**2) + (y2**2 - y1**2) + (z2**2 - z1**2)],
                  [r1**2 - r3**2 + (x3**2 - x1**2) + (y3**2 - y1**2)+ (z3**2 - z1**2)],
                  [r1**2 - r4**2 + (x4**2 - x1**2) + (y4**2 - y1**2)+ (z4**2 - z1**2)]])
    

    pos = np.linalg.lstsq(A, b)[0]



    return pos

#afstand_satellit_lstsq(satellitter, geo_afstande)

<>:16: SyntaxWarning: list indices must be integers or slices, not tuple; perhaps you missed a comma?
<>:16: SyntaxWarning: list indices must be integers or slices, not tuple; perhaps you missed a comma?
C:\Users\ebber\AppData\Local\Temp\ipykernel_41516\3406299924.py:16: SyntaxWarning: list indices must be integers or slices, not tuple; perhaps you missed a comma?
  [2*(x3 - x1), 2*(y3 - y1), 2*(z3 - z1)]


TypeError: list indices must be integers or slices, not tuple

---

## Modul 4 – Urfejlen: Den fjerde ubekendte

### Baggrund

Hidtil har vi antaget, at afstandsmålingerne er eksakte geometriske afstande. I virkeligheden måler GPS-modtageren **pseudoafstande** $P_s$: afstande beregnet ud fra transmissionstider, som er fejlbehæftede på grund af modtagerens upræcise ur.

Urfejlen $dT$ (i sekunder) medfører en systematisk afstandsfejl $w = c \cdot dT$ (i meter), der lægges til alle afstande:

$$
P_s = \underbrace{\sqrt{(X-X_s)^2 + (Y-Y_s)^2 + (Z-Z_s)^2}}_{\rho_s} + w.
$$

Nu har vi fire ubekendte: $(X, Y, Z, w)$. Vi behøver mindst **fire satellitter** for at bestemme systemet entydigt. Det ikke-lineære ligningssystem er:

$$
\sqrt{(X-X_i)^2 + (Y-Y_i)^2 + (Z-Z_i)^2} + w = P_i, \quad i = 1, 2, 3, 4.
$$

### Opgave 5 – Observationsligninger og urfejlens rolle

**a)** Forklar geometrisk, hvad det betyder at tilføje urfejlen $w$. Forestil dig, at du trækker alle kuglers radier med den samme konstant $w$ (og lader $X, Y, Z$ variere frit). Hvornår skærer de "justerede" kugler hinanden i ét punkt?

**b)** Vis, at man ikke kan eliminere de kvadratiske led ($X^2,Y^2,Z^2$) direkte i observationsligningerne ved simpel subtraktion, når urfejlen $w$ indgår. Hvad sker der med $w$-leddene, når du trækker ligning 2 fra ligning 1.

**c)** Beregn i Python forskellen mellem pseudoafstandene og de sande geometriske afstande for alle seks satellitter. Hvad er den typiske størrelse af urfejlen $w$? Svarer det til en bekymrende urfejl?

```{admonition} Mere information om b)
:class: dropdown
I delspørgsmål b) så vi at man ikke umiddelbart kan fjerne de kvadratiske led ved simpel subtraktion, som man kunne i 2D-tilfældet uden urfejl. Selvom $w$-leddene eliminerer hinanden ved direkte subtraktion, efterlades man med en ikke-lineær differens af kvadratrødder. Det er dog faktisk muligt at lave en algebraisk linearisering (som generelt kræver 5 satellitter), se [Bancrofts metode](https://sps.ewi.tudelft.nl/Education/courses/ee4c03/assignments/indoor_loc/Bancroft85loc.pdf). I Bancrofts metode løses ligningerne eksakt uden brug af iterationer (modsat Gauss-Newtons iterative metode der introduceres i næste afsnit), men Bancrofts metode har den ulempe at man ved at kvadrere afstandsligningerne også risikerer man at "kvadrere" problemets følsomhed over for støj og afrundingsfejl (små målefejl i fx pseudoafstandene kan blive forstærket). I praksis kan Bancrofts metode bruges til at finde et startgæt til Gauss-Newtons metode. Vi vil dog ikke behandle Bancrofts metode yderligere i dette projekt, men derimod fokusere på de mere anvendte iterative løsere (I er selvfølgelig velkomne til at studere og inddrage Bancrofts metode, men det er altså "uden for pensum").
```

In [ ]:
for i in range(len(geo_afstande)):
    ur_fejl = geo_afstande[i] - satellitter[i][3]
    print(ur_fejl)


1861.2238210178912
1856.353687800467
1855.0950328037143
1858.2038576640189
1855.009881477803
1858.4286807626486


---

## Modul 5 – Linearisering og Gauss-Newtons metode

### Baggrund

Det ikke-lineære system fra Modul 4 løses ved hjælp af **Gauss-Newtons metode** (en variant af Newton-Raphson for mindste kvadraters problemer). Idéen er at linearisere ligningerne ved hjælp af **første-ordens Taylor-udvikling** i et startpunkt $\mathbf{x}_0 = (X_0, Y_0, Z_0, w_0)^T$.

Lad $f_i(\mathbf{x}) = \rho_i(\mathbf{x}) + w - P_i$, hvor $\rho_i = \sqrt{(X-X_i)^2+(Y-Y_i)^2+(Z-Z_i)^2}$.

Taylor-udviklingen om $\mathbf{x}_0$ giver:

$$
f_i(\mathbf{x}_0 + \Delta\mathbf{x}) \approx f_i(\mathbf{x}_0) + \nabla f_i(\mathbf{x}_0)^T \Delta\mathbf{x}.
$$

Sætter vi dette lig nul og kalder korrektionsvektoren $\mathbf{x} = ({\bar X}, {\bar Y}, {\bar Z}, {\bar w})^T$, fås det lineære system:

$$
A \mathbf{x} = \mathbf{b}.
$$

Designmatricen $A$ og højresiden $\mathbf{b}$ er defineret som (for $n$ satellitter):

$$
A = 
\begin{pmatrix} 
 \frac{X_0-X_1}{\rho_1^0} & \frac{Y_0-Y_1}{\rho_1^0} & \frac{Z_0-Z_1}{\rho_1^0} & 1 \\ \vdots & \vdots & \vdots & \vdots \\
 \frac{X_0-X_n}{\rho_n^0} & \frac{Y_0-Y_n}{\rho_n^0} & \frac{Z_0-Z_n}{\rho_n^0} & 1 
 \end{pmatrix}, 
 \qquad \mathbf{b} = \begin{pmatrix} P_1 - \rho_1^0 - w_0 \\ \vdots \\ P_n - \rho_n^0 - w_0 \end{pmatrix}.
$$

hvor $\rho_i^0 = \sqrt{(X_0-X_i)^2+(Y_0-Y_i)^2+(Z_0-Z_i)^2}$.

Løsningen $\Delta\mathbf{x} = (\bar{X}, \bar{Y}, \bar{Z}, \bar{w})$ er en korrektion til startpunktet. Den iterative procedure er:

$$
\mathbf{x}_{k+1} = \mathbf{x}_k + \Delta\mathbf{x}_k.
$$

### Opgave 6 – Designmatricen og dens fortolkning

**a)** Vis ved at differentiere $\rho_i = \sqrt{(X-X_i)^2+(Y-Y_i)^2+(Z-Z_i)^2}$, at de partielle afledte er:

$$
\frac{\partial \rho_i}{\partial X} = \frac{X-X_i}{\rho_i}, \quad \frac{\partial \rho_i}{\partial Y} = \frac{Y-Y_i}{\rho_i}, \quad \frac{\partial \rho_i}{\partial Z} = \frac{Z-Z_i}{\rho_i}.
$$

**b)** Vis, at de tre første søjler i en række af $A$ svarer til koordinaterne af en enhedsvektor fra satellit $i$ mod modtageren. Hvad beskriver designmatricen $A$ geometrisk?

**c)** Vis, at den fjerde søjle i $A$ altid består af $1$'ere. Hvad sker der, hvis man vælger en anden startværdi for $w_0$?

Opgave 6b 
rho_i er længden er vektoren i X - X_i er en forbindelsesvektor mellem modtageren og satelitten. Hvis vi dividere, bliver det så en enhedsvektor, da den får længden 1. 

Geometrisk set repræsenterer designmatricen A, retningen satelitten pejer mod personen. 

Opgave 6c
Hvis vi pluser med w, og partielt differentiere med hensyn til w, bliver det 1. Og da w er ens for alle, bliver det altid 1. Hvis vi vælger en anden w_0, så vil vi få et andet startpunkt. 

### Opgave 7 – Implementering og én iteration

**a)** Implementér en Python-funktion `byg_Ab(x0, w0, satellitter)`, der for et givet startpunkt $(X_0, Y_0, Z_0, w_0)$ beregner designmatricen $A$ og højresiden $\mathbf{b}$.

**b)** Løs det lineære system $A\mathbf{x} = \mathbf{b}$ med de fire første satellitter (4, 14, 16, 18) og startpunktet fra $x_0$ og $w_0 = 0$. Hvad er korrektionsvektoren $\Delta\mathbf{x}$? Hvad er den opdaterede position?

**c)** Sammenlign den opdaterede position med den kendte sande position $x_{sand}$. Hvor stor er fejlen?

In [43]:
def byg_Ab(x0, w0, satellitter):
    """
    Bygger designmatricen A og højresiden b for Gauss-Newton-iterationen.
    x0:        (3,) array - nuværende positionsestimate (X0, Y0, Z0)
    w0:         float - nuværende estimate for urfejl (i meter)
    satellitter:(n x 4) array - [Xs, Ys, Zs, Ps] for n satellitter
    """
    # TILFØJ DIN KODE HER

    n = satellitter.shape[0]
    A = np.zeros((n, 4))
    b = np.zeros(n)

    for i in range(n):
        Xs, Ys, Zs, Ps = satellitter[i]

        rho = np.sqrt((Xs - x0[0])**2 + (Ys - x0[1])**2 + (Zs - x0[2])**2)
        A[i, 0] = -(Xs - x0[0]) / rho
        A[i, 1] = -(Ys - x0[1]) / rho
        A[i, 2] = -(Zs - x0[2]) / rho
        A[i, 3] = 1

        b[i] = Ps - (rho + w0)
    
    return A, b

# Opgave 7b+c – skriv din løsning her

In [89]:
#print(np.linalg.lstsq(byg_Ab(x0, 0, satellitter[:5])[0], byg_Ab(x0, 0, satellitter[:5])[1])[0])
print(byg_Ab(x0, 0, satellitter[:4]))
d_x = np.linalg.lstsq(byg_Ab(x0, 0, satellitter[:5])[0], byg_Ab(x0, 0, satellitter[:5])[1])[0]

(array([[-0.03922803,  0.70340009, -0.70971083,  1.        ],
       [-0.43940819, -0.45917879, -0.77205912,  1.        ],
       [-0.58939527,  0.3291562 , -0.73774617,  1.        ],
       [-0.93450075,  0.35218083,  0.05173976,  1.        ]]), array([-1915.17968376, -1866.28890452, -1903.78940317, -1892.34082832]))


(array([[-0.03922803,  0.70340009, -0.70971083,  1.        ],
       [-0.43940819, -0.45917879, -0.77205912,  1.        ],
       [-0.58939527,  0.3291562 , -0.73774617,  1.        ],
       [-0.93450075,  0.35218083,  0.05173976,  1.        ]]), array([-1915.17968376, -1866.28890452, -1903.78940317, -1892.34082832]))


/var/folders/sc/84m76hcx3_q4jm63jgbvp6f40000gn/T/ipykernel_16441/4105824542.py:3: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  d_x = np.linalg.lstsq(byg_Ab(x0, 0, satellitter[:5])[0], byg_Ab(x0, 0, satellitter[:5])[1])[0]


In [90]:
x1 = d_x[:3] + x0
print(x_sand - x1)

[-2.01843726  2.63647208 -1.61721324]


### Opgave 8 – Iterativ løsning (Gauss-Newton)

**a)** Implementér den fulde iterative Gauss-Newton procedure. Start fra $x_0$ og $w_0 = 0$. Gentag iterationen, indtil normen af korrektionsvektoren $\|\Delta\mathbf{x}\|$ er under et stopkriterie (fx $10^{-4}$ m).

**b)** Udskriv positionsestimatet og fejlen efter hver iteration. Hvor mange iterationer er nødvendige?

**c)** Undersøg konvergensens følsomhed over for startpunktet. Prøv at starte fra Jordens centrum $(0, 0, 0, 0)$. Hvad sker der?

In [92]:
tol = 10e-4      # skal ændres
max_iter = 2000000 # skal ændres  
def gauss_newton_gps(startpos, w_start, satellitter, tol=tol, max_iter=max_iter, verbose=True):
    """
    Gauss-Newtons metode til GPS-positionering.
    Returnerer: (pos, w, antal_iter, fejl_liste)
    """
    x = np.array(startpos, dtype=float)
    w = float(w_start)
    fejl_liste = []
    iters = 0

    while iters < max_iter:
        A, b = byg_Ab(x, w, satellitter[:4])
        d_x = np.linalg.lstsq(A, b, rcond=None)[0]

        x = x + d_x[:3]
        w = w + d_x[3]
        iters += 1

        fejl = np.linalg.norm(x - x_sand)
        fejl_liste.append(fejl)

        if verbose:
            print(f"Iteration {iters}: x={x}, w={w:.3f} m, fejl={fejl:.3f} m")

        if np.linalg.norm(d_x) < tol:
            break

    return x, w, iters, fejl_liste

print(gauss_newton_gps(x0, 0, satellitter))

Iteration 1: x=[3504309.93757308  780753.44210333 5252120.20310128], w=-1867.703 m, fejl=13.697 m
Iteration 2: x=[3504309.93758733  780753.44214708 5252120.20309181], w=-1867.703 m, fejl=13.697 m
(array([3504309.93758733,  780753.44214708, 5252120.20309181]), -1867.702709740715, 2, [13.696618831162887, 13.69661350315528])


In [93]:
jord_centrum = np.array([0, 0, 0])
print(gauss_newton_gps(jord_centrum, 0, satellitter[:4]))

Iteration 1: x=[4267775.29484209  970993.28695353 6366426.51370898], w=1389110.657 m, fejl=1364079.779 m
Iteration 2: x=[3532343.10579652  789999.61236223 5291717.10350281], w=50847.496 m, fejl=49375.958 m
Iteration 3: x=[3504345.62182763  780770.20736834 5252171.03691113], w=-1799.755 m, fejl=51.857 m
Iteration 4: x=[3504309.93763678  780753.44218721 5252120.20316777], w=-1867.703 m, fejl=13.697 m
Iteration 5: x=[3504309.93758733  780753.44214708 5252120.20309181], w=-1867.703 m, fejl=13.697 m
(array([3504309.93758733,  780753.44214708, 5252120.20309181]), -1867.702709732445, 5, [1364079.7786030015, 49375.95770374258, 51.85734929957856, 13.696527154035184, 13.696613495395221])


In [94]:
langtvæk = np.array([1000000, 1000000, 10000000])
print(gauss_newton_gps(langtvæk, 0, satellitter[:4]))

Iteration 1: x=[3198720.55915047  806645.65144133 4756208.07224382], w=189511.281 m, fejl=583094.466 m
Iteration 2: x=[3510763.96308122  779956.45324358 5261394.62668227], w=9811.870 m, fejl=11314.051 m
Iteration 3: x=[3504312.60387853  780752.98350172 5252123.67053165], w=-1863.197 m, fejl=9.514 m
Iteration 4: x=[3504309.93758777  780753.44214698 5252120.20309234], w=-1867.703 m, fejl=13.697 m
Iteration 5: x=[3504309.93758733  780753.44214708 5252120.20309182], w=-1867.703 m, fejl=13.697 m
(array([3504309.93758733,  780753.44214708, 5252120.20309182]), -1867.7027097315254, 5, [583094.46635319, 11314.05123879702, 9.513998872572897, 13.696612829482504, 13.69661349246947])


---

## Modul 6 – Overbestemte systemer: Mindste kvadraters metode

### Baggrund

I praksis kan en GPS-modtager se 8–12 satellitter ad gangen. I stedet for kun at bruge 4 satellitter kan vi udnytte *alle* tilgængelige data til at reducere støjens indflydelse og øge præcisionen.

Når der er $n > 4$ satellitter, har det lineariserede system $A\mathbf{x} = \mathbf{b}$ flere ligninger end ubekendte og er generelt **overbestemt** (ingen eksakt løsning). Vi finder i stedet den løsning, der minimerer summen af kvadrerede residualer:

$$
\min_{\mathbf{x}} \|\mathbf{b} - A\mathbf{x}\|^2.
$$

Det kan vises, at denne løsning opfylder **normalligningerne**:

$$
A^T A \mathbf{x} = A^T \mathbf{b}.
$$

### Opgave 9 – Normalligningerne

**a)** Lad $\phi(\mathbf{x}) = \|\mathbf{b} - A\mathbf{x}\|^2 = (\mathbf{b} - A\mathbf{x})^T(\mathbf{b} - A\mathbf{x})$. Vis at $\phi(\mathbf{x})$ er en kvadratisk form. Beregn gradienten $\nabla_{\mathbf{x}} \phi$ og sæt den lig nul for at udlede normalligningerne $A^T A \mathbf{x} = A^T \mathbf{b}$. 

**b)** Vis, at matricen $A^T A$ er symmetrisk. Argumentér for, at $A^T A$ er positiv semi-definit, og at den er positiv definit (og dermed invertibel), når $A$ har rang 4. Hvordan er matricen $A^T A$ relateret til Hesse-matricen for den kvadratiske form $\phi(\mathbf{x})$? Brug denne relation til en anden ordens test at de stationære punkter for $\phi(\mathbf{x})$ (under antagelse at $A^T A$ er positiv definit).  

**c)** Forklar, hvad residualvektoren $\mathbf{R} = \mathbf{b} - A\hat{\mathbf{x}}$ fortæller om kvaliteten af positionsbestemmelsen.

(exercise:gps-10)=
### Opgave 10 – GPS med alle seks satellitter

**a)** Modificér Gauss-Newton-proceduren til at bruge alle seks satellitter (det overbestemte tilfælde). Brug `np.linalg.lstsq` til at løse det overbestemte lineære system i hvert iterationstrin.

**b)** Sammenlign den opnåede præcision med løsningen baseret på kun fire satellitter. Er der en forbedring?

**c)** Beregn residualvektoren $\mathbf{R} = \mathbf{b} - A\hat{\mathbf{x}}$ for den endelige løsning. Hvad siger størrelsen af residualerne om måledatakvaliteten?

In [95]:
def byg_Ab_10(x0, w0, satellitter):
    """
    Bygger designmatricen A og højresiden b for Gauss-Newton-iterationen.
    x0:        (3,) array - nuværende positionsestimate (X0, Y0, Z0)
    w0:         float - nuværende estimate for urfejl (i meter)
    satellitter:(n x 4) array - [Xs, Ys, Zs, Ps] for n satellitter
    """
    # TILFØJ DIN KODE HER

    n = satellitter.shape[0]
    A = np.zeros((n, 4))
    b = np.zeros(n)

    for i in range(n):
        Xs, Ys, Zs, Ps = satellitter[i]

        rho = np.sqrt((Xs - x0[0])**2 + (Ys - x0[1])**2 + (Zs - x0[2])**2)
        A[i, 0] = -(Xs - x0[0]) / rho
        A[i, 1] = -(Ys - x0[1]) / rho
        A[i, 2] = -(Zs - x0[2]) / rho
        A[i, 3] = 1

        b[i] = Ps - (rho + w0)
    
    return A, b

# Opgave 7b+c – skriv din løsning her

In [96]:
tol = 10e-4      # skal ændres
max_iter = 200 # skal ændres  
def gauss_newton_gps(startpos, w_start, satellitter, tol=tol, max_iter=max_iter, verbose=True):
    """
    Gauss-Newtons metode til GPS-positionering.
    Returnerer: (pos, w, antal_iter, fejl_liste)
    """
    x = np.array(startpos, dtype=float)
    w = float(w_start)
    fejl_liste = []
    iters = 0

    while iters < max_iter:
        A, b = byg_Ab_10(x, w, satellitter)
        d_x = np.linalg.lstsq(A, b, rcond=None)[0]

        x = x + d_x[:3]
        w = w + d_x[3]
        iters += 1

        fejl = np.linalg.norm(x - x_sand)
        fejl_liste.append(fejl)

        if verbose:
            print(f"Iteration {iters}: x={x}, w={w:.3f} m, fejl={fejl:.3f} m")

        if np.linalg.norm(d_x) < tol:
            break

    return x, w, iters, fejl_liste

print(gauss_newton_gps(x0, 0, satellitter))

Iteration 1: x=[3504320.55238356  780753.48391236 5252128.77070908], w=-1857.409 m, fejl=0.058 m
Iteration 2: x=[3504320.55238067  780753.4839505  5252128.77068363], w=-1857.409 m, fejl=0.058 m
(array([3504320.55238067,  780753.4839505 , 5252128.77068363]), -1857.40908843923, 2, [0.05817298182568664, 0.05817762913056242])


In [125]:
x_hat, w_0 = gauss_newton_gps(x0, 0, satellitter)[:2]
A , b = byg_Ab_10(x_hat, w_0, satellitter)

d_x_hat = np.linalg.lstsq(A, b, rcond=None)[0]
R = b - A @ d_x_hat
print(R)
print(np.linalg.norm(R))

Iteration 1: x=[3504320.55238356  780753.48391236 5252128.77070908], w=-1857.409 m, fejl=0.058 m
Iteration 2: x=[3504320.55238067  780753.4839505  5252128.77068363], w=-1857.409 m, fejl=0.058 m
[-3.8261175   1.00447278  2.26964373 -0.83210039  2.42289264 -1.03879127]
5.33303951099682


### Opgave 11 – Spredningsanalyse (**)

Når der er mere data end ubekendte, kan vi estimere usikkerheden på positionen via **varians-kovariansmatricen**:

$$
Q_x = \sigma_0^2 \, (A^T A)^{-1}, \quad \sigma_0^2 = \frac{\mathbf{R}^T \mathbf{R}}{n - m},
$$

hvor $n$ er antallet af satellitter og $m = 4$ er antallet af ubekendte.

**a)** Beregn $\sigma_0^2$ og $Q_x$ for løsningen med 6 satellitter.

**b)** Spredningerne på koordinaterne er $\sigma_X = \sqrt{[Q_x]_{11}}$, $\sigma_Y = \sqrt{[Q_x]_{22}}$, $\sigma_Z = \sqrt{[Q_x]_{33}}$. Beregn disse og sammenlign med den faktiske fejl.

**c)** Hvad fortæller $\sigma_0$ om kvaliteten af pseudoafstandsmålingerne?

In [140]:
n = A.shape[0]
m = A.shape[1]
sigma_0_squared = (R.T @ R) / (n - m)
print(sigma_0_squared)
print(n,m)

14.220655212926603
6 4


In [141]:
Q_x = sigma_0_squared * np.linalg.inv(A.T @ A)
print(Q_x)

[[11.68952952 -1.71740746  7.34040458  6.74918718]
 [-1.71740746  8.62781116 -6.65972088 -5.39256159]
 [ 7.34040458 -6.65972088 36.04141002 21.94937458]
 [ 6.74918718 -5.39256159 21.94937458 16.46042698]]


In [143]:
sigma_x = np.sqrt(Q_x[0][0])
sigma_y = np.sqrt(Q_x[1][1])
sigma_z = np.sqrt(Q_x[2][2])
print(sigma_x, sigma_y, sigma_z)

faktisk_fejl = np.linalg.norm(x_hat - x_sand)
print(faktisk_fejl)

3.418995396790141 2.9373135955711756 6.003449843039578
0.05817762913056242


---

## Modul 7 – Perspektivering

### 7.1 Relativitetsteorien og GPS

En faktor, der *ikke* indgår i vores simple model men er afgørende i praksis, er **Einsteins relativitetsteori**. GPS-satellitter befinder sig ca. 20.200 km over Jordens overflade og bevæger sig med ca. 3,87 km/s. To relativistiske effekter påvirker satellitternees ure:

* **Speciel relativitetsteori:** Satellittens hastighed får dens ur til at gå *langsommere* med ca. $-7\ \mu\text{s/dag}$.
* **Generel relativitetsteori:** Den svagere tyngdekraft i banen får uret til at gå *hurtigere* med ca. $+45\ \mu\text{s/dag}$.

Den samlede effekt er $+38\ \mu\text{s/dag}$: satellitterne ure *springer frem* ca. 38 mikrosekunder per dag i forhold til ure på Jordens overflade.

### Opgave 12 – Relativistisk positionsfejl (*)

**a)** Beregn den akkumulerede positionsfejl efter 1 time, hvis man ignorerede den relativistiske korrektion på $+38\ \mu\text{s/dag}$.

**b)** Hvad ville fejlen være efter blot 1 minut? Diskutér, hvornår GPS-systemet ville have givet fuldstændig meningsløse positioner.

### 7.2 GDOP – Geometrisk fortyndning af præcision

Designmatricen $A$ beskriver ikke blot lineariseringen, men også geometrien i satellitkonstellationen. Hvis alle satellitterne er samlet i én del af himlen (dårlig geometri), er $A$ dårligt konditioneret og løsningen upræcis.

**GDOP** (Geometric Dilution of Precision) er et mål for denne geometriske effekt:

$$
\text{GDOP} = \sqrt{\text{tr}((A^T A)^{-1})}.
$$

En lav GDOP (tæt på 1) er god; en høj GDOP (> 6) giver unøjagtige positioner.

### Opgave 13 – GDOP-beregning (*)

**a)** Beregn GDOP for den aktuelle satellitgeometri (4 og 6 satellitter).

**b)** Konstruér et kunstigt eksempel med 4 satellitter, der alle er samlet på den samme side af himlen (tæt på hinanden). Vis, at GDOP stiger.

In [ ]:
def beregn_GDOP(A):
    """Beregner GDOP fra designmatricen A."""

# TILFØJ DIN KODE HER

raise NotImplementedError("Implementer beregn_GDOP for at evaluere geometrisk kvalitet af satellitkonfigurationen.")

NotImplementedError: Implementer beregn_GDOP for at evaluere geometrisk kvalitet af satellitkonfigurationen.

---

## Modul 8 – Ekstra undersøgelser (valgfrit)

### 8.1 Alle kombinationer af fire satellitter (**)

### Opgave 14

Brug alle $\binom{6}{4} = 15$ kombinationer af fire satellitter ud af de seks tilgængelige. Beregn en position for hver kombination og sammenlign resultaterne.

**a)** Beregn gennemsnit og spredning på de 15 positionsestimater.

**b)** Diskutér, om denne metode er bedre eller dårligere end mindste kvadraters metode med alle seks satellitter.

**c)** Er der nogen kombinationer, der giver markant dårligere resultater? Undersøg sammenhængen med GDOP.

### 8.2 Visualisering af satellitpositioner (**)

### Opgave 15

GPS-beregninger sker i 3D kartesiske koordinater (WGS84), men satellitpositioner visualiseres normalt i **azimut-elevationsplot** (himmelkort).

**a)** Konvertér satelliternes WGS84-positioner til lokale horisontale koordinater (azimut, elevation) set fra modtagerens position.

**b)** Visualisér satelliternes position på et polart plot ("sky plot"). Diskutér, hvad figuren fortæller om GDOP.

---

## Appendiks A – GPS-satellitsystemet

### Det matematiske minimum (24 satellitter)

Da GPS-systemet blev designet af det amerikanske forsvar, var det baseret på præcis **24 satellitter** fordelt på 6 baner med 4 satellitter i hver. Dette antal er det matematiske minimum for at sikre, at en bruger overalt på Jorden altid kan "se" mindst 4 satellitter – nødvendigt for at bestemme en 3D-position og korrigere for urfejl.

### Den nuværende situation

I 2024 er der typisk **30–32 aktive GPS-satellitter**. Ekstra satellitter fungerer som reserver og øger præcisionen.

### Andre systemer (GNSS)

Moderne telefoner bruger ofte alle tilgængelige systemer (GNSS):

| System  | Land  | Ca. antal satellitter |
|---------|-------|-----------------------|
| GPS     | USA   | 31                    |
| Galileo | EU    | 28                    |
| GLONASS | Rusland | 24                  |
| BeiDou  | Kina  | 35                    |

## Appendiks B – WGS84 koordinatsystemet

Alle GPS-beregninger foregår i **WGS84** (World Geodetic System 1984):

* **Origo:** Jordens massemidtpunkt
* **$x$-akse:** Mod skæringen af Ækvatoren og Greenwichmeridianen
* **$z$-akse:** Mod geografisk nordpol (Jordens rotationsakse)
* **$y$-akse:** Vinkelret på $x$ og $z$ (højrehåndssystem)

Koordinaterne i WGS84 omregnes ofte til bredde- og længdegrader for præsentation, men alle beregninger sker i kartesiske $(X, Y, Z)$-koordinater.

## Litteratur og videre læsning

* **GPS-bogen:** K. Dueholm, M. Laurentzius, A. Jensen: *GPS*, Nyt Teknisk Forlag, 2005.
* **SIAM-artikel:** G. Nord, D. Jabon, J. Nord: *The global positioning system and the implicit function theorem*, SIAM Rev., 40(3), 1998.
* **Mindste kvadrater:** Se kursusmaterialet i Mat1b, kapitel 4.
* **Taylor-udvikling:** Se kursusmaterialet i Mat1b, kapitel 6.
* **NumPy dokumentation:** [numpy.org](https://numpy.org)
* **SciPy dokumentation:** [scipy.org](https://scipy.org)